# 00. Input data

Two datasets feed every analysis. Each is built and verified by its own notebook, with the
code inside it; both write to this `python/` folder.

| Order | Notebook | Builds | Time |
|---|---|---|---|
| 1 | `simCellsSpace/simulate_cells.ipynb` | `sim_cells_datasets_co_mo_simulated_classical.h5ad`: synthetic tissue with a known network (seed 42) | ~15 s |
| 2 | `data_prep/build_ovarian.ipynb` | `dataset_co_mo2.h5ad`: ovarian cancer + fibroblast mono/co-culture from GEO GSE224333 (needs internet) | minutes |

Each notebook checks its dataset against stored reference histograms
(`simCellsSpace/reference_histograms.csv`, `data_prep/reference_histograms.csv`) and saves
its histogram figure to `results/data/`. For the R tools (CellChat) convert with
`Rscript data_prep/convert_h5ad_rds.R h5ad2rds dataset_co_mo2.h5ad dataset_co_mo2.rds`.

The cell below only checks that both datasets are present and complete.

In [ ]:
# Work from the QuantumXCT/python folder wherever Jupyter was started: the dataset is
# written to python/ and the figures to python/results/data/.
import os, sys
_root = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(_root, "qxct")) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
print("working directory:", _root)

In [ ]:
import pandas as pd
import anndata as ad

DATASETS = [
    ("synthetic", "sim_cells_datasets_co_mo_simulated_classical.h5ad", "cell_type",
     "simCellsSpace/simulate_cells.ipynb", "simCellsSpace/reference_histograms.csv", "seed42",
     {"ct1_mo": "Type_A_Lone", "ct2_mo": "Type_B_Lone", "ct1_co": "Type_A_Interact", "ct2_co": "Type_B_Interact"}),
    ("ovarian", "dataset_co_mo2.h5ad", "BatchID",
     "data_prep/build_ovarian.ipynb", "data_prep/reference_histograms.csv", "reference",
     {"ct1_mo": "Cancer Cells (Mo)", "ct2_mo": "Fibroblasts (Mo)", "ct1_co": "Cancer Cells (Co)", "ct2_co": "Fibroblasts (Co)"}),
]
rows = []
for name, path, col, notebook, ref_path, ref_col, groups in DATASETS:
    if not os.path.exists(path):
        rows.append({"dataset": name, "file": path, "status": f"MISSING -> run {notebook}"})
        continue
    a = ad.read_h5ad(path, backed="r")
    ref = pd.read_csv(ref_path).query("item == 'n_cells'").set_index("group")[ref_col]
    n = {k: int((a.obs[col].astype(str) == v).sum()) for k, v in groups.items()}
    ok = all(n[k] == ref[k] for k in groups)
    rows.append({"dataset": name, "file": path, "cells x genes": f"{a.n_obs} x {a.n_vars}",
                 "cells per group": " / ".join(str(n[k]) for k in groups),
                 "status": "OK" if ok else f"group sizes differ from reference -> re-run {notebook}"})
pd.DataFrame(rows)